# Datenerfassung DB Getränkebestllung

In welcher Reihenfolge befüllen wir die Tabellen?

- Stammdaten
  - Lieferant
  - Gebinde
  - Artikel
  - Sortiment
- Transaktionen für die Bestellung
  - Bestellung
  - Bestellposten


In [3]:
-- LIEFERANT
INSERT INTO LIEFERANT (Firma, Strasse, PLZ, Ort)
VALUES ('Schiller KG', 'Goetheplatz 3', '12345', 'Berlin'),
       ('Maurer GmbH', 'Bahnweg 45', '02345', 'Potsdam'),
       ('Bäcker GmbH', 'Platz 45', '65623', 'Reuth');

(3 rows affected)

Total execution time: 00:00:00.007

In [4]:
SELECT * FROM LIEFERANT;

(3 rows affected)

LieferantID | Firma       | Strasse       | PLZ   | Ort    
------------+-------------+---------------+-------+--------
1           | Schiller KG | Goetheplatz 3 | 12345 | Berlin 
2           | Maurer GmbH | Bahnweg 45    | 02345 | Potsdam
3           | Bäcker GmbH | Platz 45      | 65623 | Reuth  
(3 rows)

Total execution time: 00:00:00.098

In [5]:
-- GEBINDE
INSERT INTO GEBINDE (GebindeArt)
VALUES ('Fass'),
('Karton'),
('Kiste'),
('Six-Pack');

(4 rows affected)

Total execution time: 00:00:00.004

In [6]:
SELECT * FROM GEBINDE;

(4 rows affected)

GebindeID | GebindeArt
----------+-----------
1         | Fass      
2         | Karton    
3         | Kiste     
4         | Six-Pack  
(4 rows)

Total execution time: 00:00:00.021

In [7]:
-- ARTIKEL
INSERT INTO ARTIKEL (Artikel, UVP, GebindeID)
VALUES ('Hydro-Wasser', 7.99, 3),
('Cola-Mix-Spree', 11.99, 3),
('Stone-Energy-Drink', 22, 2),
('Weizen Hell Rhein-Bräu', 13.99, 3);

(4 rows affected)

Total execution time: 00:00:00.043

In [12]:
SELECT a.ArtikelID, a.Artikel, a.UVP, g.GebindeArt
FROM GEBINDE AS g
JOIN ARTIKEL AS a ON g.GebindeID = a.GebindeID;

(4 rows affected)

ArtikelID | Artikel                | UVP   | GebindeArt
----------+------------------------+-------+-----------
1         | Hydro-Wasser           | 7.99  | Kiste     
2         | Cola-Mix-Spree         | 11.99 | Kiste     
3         | Stone-Energy-Drink     | 22.00 | Karton    
4         | Weizen Hell Rhein-Bräu | 13.99 | Kiste     
(4 rows)

Total execution time: 00:00:00.021

In [9]:
-- SORTIMENT
INSERT INTO SORTIMENT (LieferantID, ArtikelID, Angebotspreis)
VALUES (1, 1, 7.99),
(1, 2, 11.99),
(1, 4, 13.99),
(2, 3, 22),
(3, 1, 5.46),
(3, 3, 20);

(6 rows affected)

Total execution time: 00:00:00.013

In [17]:
SELECT l.Firma, a.Artikel, s.Angebotspreis, a.UVP
FROM LIEFERANT AS l
JOIN SORTIMENT AS s ON l.LieferantID = s.LieferantID
JOIN ARTIKEL AS a ON s.ArtikelID = a.ArtikelID;

(6 rows affected)

Firma       | Artikel                | Angebotspreis | UVP  
------------+------------------------+---------------+------
Schiller KG | Hydro-Wasser           | 7.99          | 7.99 
Schiller KG | Cola-Mix-Spree         | 11.99         | 11.99
Schiller KG | Weizen Hell Rhein-Bräu | 13.99         | 13.99
Maurer GmbH | Stone-Energy-Drink     | 22.00         | 22.00
Bäcker GmbH | Hydro-Wasser           | 5.46          | 7.99 
Bäcker GmbH | Stone-Energy-Drink     | 20.00         | 22.00
(6 rows)

Total execution time: 00:00:00.008

In [ ]:
-- Transaktion für die erste Bestellung
BEGIN TRANSACTION;
BEGIN TRY
  INSERT INTO BESTELLUNG (Datum, LieferantID)
    VALUES ('10.02.2016', 1);
  DECLARE @BestellungID INT = @@IDENTITY;
  INSERT INTO BESTELLPOSTEN (BestellungID, ArtikelID, Menge, EInzelpreis)
    VALUES (@BestellungID, 1, 5, 7.99);
INSERT INTO BESTELLPOSTEN (BestellungID, ArtikelID, Menge, EInzelpreis)
    VALUES (@BestellungID, 2, 5, 11.99);
  COMMIT TRANSACTION;
  PRINT 'Prima! Bestellung ist beauftragt.';
END TRY
BEGIN CATCH
  ROLLBACK TRANSACTION;
  PRINT 'Schade, das hat leider nicht geklappt!';
  THROW;
END CATCH;

Msg 547, Level 16, State 0, Line 7
Die INSERT-Anweisung steht in Konflikt mit der FOREIGN KEY-Einschränkung "fk_BESTELLPOSTEN_ARTIKEL". Der Konflikt trat in der Getraenkebestellung-Datenbank, Tabelle "dbo.ARTIKEL", column 'ArtikelID' auf.

(1 row affected)
(0 rows affected)
Schade, das hat leider nicht geklappt!

Total execution time: 00:00:00.041

In [ ]:
-- Transaktion für die zweite Bestellung mit Fehler
BEGIN TRANSACTION;
BEGIN TRY
  INSERT INTO BESTELLUNG (Datum, LieferantID)
    VALUES ('17.02.2016', 2);
  DECLARE @BestellungID INT = @@IDENTITY;
  INSERT INTO BESTELLPOSTEN (BestellungID, ArtikelID, Menge, EInzelpreis)
    VALUES (@BestellungID, 33, 10, 22);
  COMMIT TRANSACTION;
  PRINT 'Prima! Bestellung ist beauftragt.';
END TRY
BEGIN CATCH
  ROLLBACK TRANSACTION;
  PRINT 'Schade, das hat leider nicht geklappt!';
  THROW;
END CATCH;

In [7]:
-- Transaktion für die zweite Bestellung ohne Fehler
BEGIN TRANSACTION;
BEGIN TRY
  INSERT INTO BESTELLUNG (Datum, LieferantID)
    VALUES ('17.02.2016', 2);
  DECLARE @BestellungID INT = @@IDENTITY;
  INSERT INTO BESTELLPOSTEN (BestellungID, ArtikelID, Menge, EInzelpreis)
    VALUES (@BestellungID, 3, 10, 22);
  COMMIT TRANSACTION;
  PRINT 'Prima! Bestellung ist beauftragt.';
END TRY
BEGIN CATCH
  ROLLBACK TRANSACTION;
  PRINT 'Schade, das hat leider nicht geklappt!';
  THROW;
END CATCH;

(1 row affected)
(1 row affected)
Prima! Bestellung ist beauftragt.

Total execution time: 00:00:00.014

In [5]:
-- Transaktion für die dritte Bestellung
BEGIN TRANSACTION;
BEGIN TRY
  INSERT INTO BESTELLUNG (Datum, LieferantID)
    VALUES ('24.02.2016', 3);
  DECLARE @BestellungID INT = @@IDENTITY;
  INSERT INTO BESTELLPOSTEN (BestellungID, ArtikelID, Menge, EInzelpreis)
    VALUES (@BestellungID, 1, 5, 5.46),
    (@BestellungID, 3, 4, 20);
  COMMIT TRANSACTION;
  PRINT 'Prima! Bestellung ist beauftragt.';
END TRY
BEGIN CATCH
  ROLLBACK TRANSACTION;
  PRINT 'Schade, das hat leider nicht geklappt!';
  THROW;
END CATCH;

(1 row affected)
(2 rows affected)
Prima! Bestellung ist beauftragt.

Total execution time: 00:00:00.016

In [8]:
SELECT * FROM BESTELLUNG;

(3 rows affected)

BestellungID | Datum      | LieferantID
-------------+------------+------------
4            | 2016-02-10 | 1          
6            | 2016-02-24 | 3          
7            | 2016-02-17 | 2          
(3 rows)

Total execution time: 00:00:00.029